# ADM 3+1 Background

Introduce ADM fields as spatial-slice variables used by numerical relativity.

Navigation:
[Index](../index.ipynb) |
Previous: [Code-Writing Path Choice Guide](../5-infrastructures/backend_choice_guide.ipynb) |
Next: [ADM-BSSN Conversions](adm_bssn_conversions.ipynb)

## Learning Goals

- Interpret ADM variables as spatial-slice geometry plus slice-change information.
- Inspect simple analytic initial-data sets.
- Use residuals to check basic geometric identities.

## Words for This Notebook

| Name | Plain Meaning | Code Name |
| --- | --- | --- |
| spatial metric | distances within one time slice | `gammaDD` |
| extrinsic curvature | how the slice bends through spacetime | `KDD` |
| lapse | how fast time advances between slices | `alpha` |
| shift | how spatial coordinates slide between slices | `betaU` |
| shift derivative | data passed to later shift evolution equations | `BU` |
| conformally flat | equal to a scale factor times a flat metric | `psi**4 delta_ij` |
| residual | expression that should simplify to zero | `sp.simplify(...)` |

Use the code cells actively: predict what should happen, run the cell, then explain
the output in plain language.

## Step 1: Connect the ADM Fields to the Line Element

The ADM split follows Arnowitt, Deser, and Misner's
[space-plus-time formulation](https://arxiv.org/abs/gr-qc/0405109). In plain
language, it writes spacetime using spatial distances, time advance, and
coordinate sliding:

$$
ds^2 =
-\alpha^2 dt^2
+ \gamma_{ij}\left(dx^i + \beta^i dt\right)
             \left(dx^j + \beta^j dt\right).
$$

The next cells inspect concrete ADM data sets and check simple identities in the
fields that NRPy stores.

## Setup: Import Tools for ADM Checks

SymPy simplifies symbolic residuals. The NRPy initial-data classes provide
analytic ADM fields in Cartesian or spherical coordinates.

In [1]:
import sympy as sp

from nrpy.equations.general_relativity.InitialData_Cartesian import (
    InitialData_Cartesian,
)
from nrpy.equations.general_relativity.InitialData_Spherical import (
    InitialData_Spherical,
)

## Step 2: Inspect Kasner Initial Data

Kasner's exact solution ([1921](https://doi.org/10.2307/2370192)) stretches the
coordinate axes at different rates. In ADM form, the nonzero components are

$$
\gamma_{ij} =
\mathrm{diag}(t^{2p_1}, t^{2p_2}, t^{2p_3}),
\qquad
K_{ij} =
\mathrm{diag}(-p_1 t^{2p_1-1}, -p_2 t^{2p_2-1}, -p_3 t^{2p_3-1}).
$$

The next cell loads NRPy's symbolic representation and prints the fields that
should be nonzero.

In [2]:
kasner = InitialData_Cartesian("Kasner")
print("Kasner gammaDD diagonal:", [kasner.gammaDD[i][i] for i in range(3)])
print("Kasner KDD diagonal:", [kasner.KDD[i][i] for i in range(3)])
print("Kasner alpha:", kasner.alpha)
print("Kasner betaU:", kasner.betaU)
print("Kasner BU:", kasner.BU)

Kasner gammaDD diagonal: [KASNER_t0**(2*KASNER_p1), KASNER_t0**(2*KASNER_p2), KASNER_t0**(2*KASNER_p3)]
Kasner KDD diagonal: [-KASNER_p1*KASNER_t0**(2*KASNER_p1 - 1), -KASNER_p2*KASNER_t0**(2*KASNER_p2 - 1), -KASNER_p3*KASNER_t0**(2*KASNER_p3 - 1)]
Kasner alpha: 1
Kasner betaU: [0, 0, 0]
Kasner BU: [0, 0, 0]


## Validation Check: Check Kasner Diagonal Structure

The displayed Kasner formula says every off-diagonal metric and curvature entry
should vanish. These residuals compare NRPy's symbolic fields with that identity.

In [3]:
kasner_gamma_offdiag = [
    sp.simplify(kasner.gammaDD[i][j])
    for i in range(3)
    for j in range(3)
    if i != j
]
kasner_K_offdiag = [
    sp.simplify(kasner.KDD[i][j])
    for i in range(3)
    for j in range(3)
    if i != j
]
print("Kasner gammaDD off-diagonal residuals:", kasner_gamma_offdiag)
print("Kasner KDD off-diagonal residuals:", kasner_K_offdiag)
if kasner_gamma_offdiag != [0] * 6 or kasner_K_offdiag != [0] * 6:
    raise RuntimeError("Expected Kasner off-diagonal ADM residuals to vanish.")

Kasner gammaDD off-diagonal residuals: [0, 0, 0, 0, 0, 0]
Kasner KDD off-diagonal residuals: [0, 0, 0, 0, 0, 0]


## Validation Check: Check Brill-Lindquist Isotropy

The [Brill-Lindquist construction](https://doi.org/10.1103/PhysRev.131.471) is
conformally flat:

$$
\gamma_{ij} = \psi^4 \delta_{ij},
\qquad
K_{ij}=0.
$$

Therefore the three diagonal metric components should match, and every curvature
component should vanish.

In [4]:
brill_lindquist = InitialData_Cartesian("BrillLindquist")
brill_metric_residuals = [
    sp.simplify(
        brill_lindquist.gammaDD[i][i]
        - brill_lindquist.gammaDD[0][0]
    )
    for i in range(3)
]
brill_curvature_residuals = [
    sp.simplify(brill_lindquist.KDD[i][j])
    for i in range(3)
    for j in range(3)
]
print("Brill-Lindquist metric isotropy residuals:", brill_metric_residuals)
print("Brill-Lindquist KDD residuals:", brill_curvature_residuals)
print("Brill-Lindquist betaU:", brill_lindquist.betaU)
print("Brill-Lindquist BU:", brill_lindquist.BU)
if brill_metric_residuals != [0, 0, 0] or brill_curvature_residuals != [0] * 9:
    raise RuntimeError("Expected Brill-Lindquist isotropy and KDD residuals to vanish.")

Setting up reference_metric[Cartesian]...
Brill-Lindquist metric isotropy residuals: [0, 0, 0]
Brill-Lindquist KDD residuals: [0, 0, 0, 0, 0, 0, 0, 0, 0]
Brill-Lindquist betaU: [0, 0, 0]
Brill-Lindquist BU: [0, 0, 0]


## Validation Check: Check Static-Trumpet Shift Data

Dennison and Baumgarte's
[static trumpet data](https://arxiv.org/abs/1403.5484) use

$$
\alpha=\frac{r}{r+M},
\qquad
\beta^r=\frac{Mr}{(r+M)^2},
\qquad
\beta^\theta=\beta^\phi=0.
$$

The validation checks that NRPy's spherical data expose the nonzero radial shift
and zero angular shift directly.

In [5]:
static_trumpet = InitialData_Spherical("StaticTrumpet")
static_angular_shift_residuals = [
    sp.simplify(static_trumpet.betaU[1]),
    sp.simplify(static_trumpet.betaU[2]),
]
static_BU_residuals = [sp.simplify(component) for component in static_trumpet.BU]
print("Static trumpet alpha:", static_trumpet.alpha)
print("Static trumpet radial shift:", static_trumpet.betaU[0])
print("Static trumpet angular shift residuals:", static_angular_shift_residuals)
print("Static trumpet BU residuals:", static_BU_residuals)
if static_angular_shift_residuals != [0, 0] or static_BU_residuals != [0, 0, 0]:
    raise RuntimeError("Expected static-trumpet angular shift and BU residuals to vanish.")

Static trumpet alpha: r/(M + r)
Static trumpet radial shift: M*r/(M + r)**2
Static trumpet angular shift residuals: [0, 0]
Static trumpet BU residuals: [0, 0, 0]


The printed fields and explicit residual checks verify the ADM structure used by
these example data sets. These variables are the geometric inputs that the next
notebook converts into BSSN form.

## Learning Check

After running the validation cells, identify which printed residuals encode metric
isotropy, which encode zero curvature, and which encode shift structure.

## Continue

- [Indexed Expressions](../1-intro/indexedexp.ipynb)
- [Reference Metrics](../1-intro/reference_metric.ipynb)
- [ADM and BSSN Conversions](adm_bssn_conversions.ipynb)